# DalMax -- Google Colab Pro runbook (notebook form)

Runnable companion to [`COLAB_RUNBOOK.md`](https://github.com/MarioCarvalhoBr/dalmax-framework-deep-active-learning-python/blob/main/COLAB_RUNBOOK.md),
section 6 ("The clean campaign on `daninhas_full_fixed`", ADR 0016). **`COLAB_RUNBOOK.md` is the
source of truth.** This notebook is generated by `scripts/colab/build_colab_notebook.py`; do not edit
it by hand.

One Colab account runs ONE GPU part of the campaign (both manifests: the main campaign and the
KMH sweep): account A `GPU_PART = "part_1_gpu"`, account B `GPU_PART = "part_2_gpu"`, in parallel.

**Never run this notebook on the local no-GPU dev notebook** (`.claude/rules/data-safety.md`).

Fill in the **Parameters** cell right below (the only cell you edit), then run the cells top to
bottom. The gate cells raise on any problem, so "Run all" stops there instead of spending credits.

In [ ]:
import os

# Parameters -- the ONLY cell in this notebook you edit.
DRIVE_ROOT = "/content/drive/MyDrive/<FOLDER_THAT_HOLDS_DATA>"  # REQUIRED, per account: the Drive folder holding DATA/daninhas_full_fixed/
REPO_URL = "https://github.com/MarioCarvalhoBr/dalmax-framework-deep-active-learning-python.git"
REPO_DIR = "/content/dalmax"
BRANCH = "main"  # must contain ADR 0016 (the clean campaign on daninhas_full_fixed)
GPU_PART = "part_1_gpu"  # account A: "part_1_gpu"; account B: "part_2_gpu" (ADR 0016). Re-running resumes.

# Expected size of each GPU part over BOTH manifests (computed from the manifests when the notebook
# was generated); the gate cell before the run asserts it.
EXPECTED = {"part_1_gpu": (282, 94), "part_2_gpu": (177, 59)}  # (runs, run groups)
RESULTS_ROOTS = ['results/campaign_fixed', 'results/campaign_kmh_fixed']
DATASET = "daninhas_full_fixed"
DATASET_FILES = 8240

# --- gate: fail fast on a wrong value ---
if GPU_PART not in EXPECTED:
    raise ValueError(f"GPU_PART must be one of {sorted(EXPECTED)}, got {GPU_PART!r}")
if "<" in DRIVE_ROOT or not DRIVE_ROOT.startswith("/content/drive/"):
    raise ValueError("Fill in DRIVE_ROOT: the folder of THIS account's Drive that holds DATA/" + DATASET)
OTHER_GPU_PART = next(p for p in EXPECTED if p != GPU_PART)
EXPECTED_RUNS, EXPECTED_GROUPS = EXPECTED[GPU_PART]

# Export for every `!`-shell cell and for scripts/colab/setup_colab.sh (DRIVE_ROOT is required there).
os.environ["DRIVE_ROOT"] = DRIVE_ROOT
os.environ["GPU_PART"] = GPU_PART

print("DRIVE_ROOT:", DRIVE_ROOT)
print("REPO_URL:  ", REPO_URL)
print("BRANCH:    ", BRANCH)
print()
print("#" * 78)
print(f"#  GPU PART: {GPU_PART}  ->  expect {EXPECTED_RUNS} runs ({EXPECTED_GROUPS} run groups)")
print("#  Account A runs part_1_gpu; account B runs part_2_gpu. Wrong account? Edit GPU_PART, re-run.")
print("#" * 78)

## 0. Runtime check

See `COLAB_RUNBOOK.md` section 0. Select a GPU runtime (any); the cell below lists whichever GPU
Colab assigned. Its name is recorded automatically in every run's `run_metadata.json` (`environment.gpus`).

In [ ]:
!nvidia-smi

In [ ]:
import sys

# Informational only: Colab's SYSTEM Python may be 3.13; the project runs in a Poetry venv on a
# uv-managed Python 3.12 (section 3), validated right after the install.
print(f"Kernel (system) Python: {sys.version.split()[0]} -- any version is OK here")

## 1. Mount Google Drive

See `COLAB_RUNBOOK.md` section 1. The dataset must be on THIS account's Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Gate: the canonical dataset (folder or its zip) is on this Drive; the old daninhas_full is not used.
folder = f"{DRIVE_ROOT}/DATA/{DATASET}"
zip_path = f"{DRIVE_ROOT}/DATA/{DATASET}.zip"
if not (os.path.isdir(folder) or os.path.isfile(zip_path)):
    raise FileNotFoundError(
        f"Neither {folder} nor {zip_path} exists. Upload DATA/{DATASET}/ to this account's Drive under "
        "DRIVE_ROOT (COLAB_RUNBOOK.md section 6, step 1) or fix DRIVE_ROOT in the Parameters cell."
    )
print("Dataset found on Drive:", folder if os.path.isdir(folder) else zip_path)

## 2. Clone or pull the repo

Clones into `/content/dalmax` (local runtime disk, not Drive) or pulls `BRANCH` if this runtime already
has it. The gate refuses a checkout with modified tracked files. See `COLAB_RUNBOOK.md` section 2.

In [ ]:
import subprocess

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"{REPO_DIR} already exists -- pulling {BRANCH}...")
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    print(f"Cloning {REPO_URL} (branch {BRANCH}) into {REPO_DIR}...")
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)

# Gate: no modified tracked file (results/ symlink and DATA/ are untracked and do not count).
dirty = subprocess.run(["git", "-C", REPO_DIR, "status", "--porcelain", "--untracked-files=no"],
                       check=True, capture_output=True, text=True).stdout.strip()
if dirty:
    raise RuntimeError("The checkout has local modifications; reset them before running:\n" + dirty)

os.chdir(REPO_DIR)
%cd $REPO_DIR
!git log --oneline -1

## 3. Install dependencies with Poetry

See `COLAB_RUNBOOK.md` section 3.

In [ ]:
# Colab's system Python may be 3.13, which torch==2.5.0 has no wheels for.
# Install a separate Python 3.12 with uv and point Poetry at it (idempotent).
!pip install -q uv poetry
!uv python install 3.12

In [ ]:
%cd {REPO_DIR}
!poetry env use "$(uv python find 3.12)"
!poetry install

In [ ]:
import subprocess

# Gate: the Poetry venv interpreter (not the kernel's own Python) is 3.12 and sees a GPU.
probe = subprocess.run(
    ["poetry", "run", "python", "-c",
     "import sys, torch; print(sys.version.split()[0], torch.__version__, torch.cuda.is_available())"],
    capture_output=True, text=True, cwd=REPO_DIR,
)
print(probe.stdout.strip(), probe.stderr.strip()[-500:])
fields = probe.stdout.split()
venv_version = fields[0] if probe.returncode == 0 and fields else ""
if not venv_version.startswith("3.12"):
    raise RuntimeError(
        f"Poetry venv Python is {venv_version or 'unavailable'}, expected 3.12.x: re-run the previous "
        'cells (`uv python install 3.12`, `poetry env use "$(uv python find 3.12)"`, `poetry install`).'
    )
if fields[-1:] != ["True"]:
    raise RuntimeError("torch sees no GPU: Runtime -> Change runtime type -> GPU, then start again.")
print(f"Poetry venv Python {venv_version} OK, CUDA available")

## 4. Dataset and results/ (Drive)

`make colab-setup` runs `scripts/colab/setup_colab.sh`: it builds `daninhas_full_fixed.zip` on Drive on the
first session only, unzips it onto the local runtime disk, checks the file count (8240)
and that no MD5 repeats (`scripts/datasets/build_daninhas_full_fixed.py --check`), and symlinks
`results/` to `$DRIVE_ROOT/results`. Idempotent. See `COLAB_RUNBOOK.md` section 4.

In [ ]:
import subprocess

subprocess.run(["make", "colab-setup"], cwd=REPO_DIR, check=True)  # raises on any setup failure

In [ ]:
import subprocess

# Gate: the local dataset copy is the canonical one.
n_files = sum(len(files) for _, _, files in os.walk(f"{REPO_DIR}/DATA/{DATASET}"))
if n_files != DATASET_FILES:
    raise RuntimeError(f"DATA/{DATASET} has {n_files} files, expected {DATASET_FILES}")
subprocess.run(["make", "dataset-check"], cwd=REPO_DIR, check=True)
print(f"DATA/{DATASET}: {n_files} files, no repeated MD5")

## 5. Sanity checks

See `COLAB_RUNBOOK.md` section 5: the fast tests, then one short real-data GPU run into
`results/colab_check/` (never part of the campaign trees).

In [ ]:
import subprocess

subprocess.run(["make", "test"], cwd=REPO_DIR, check=True)

In [ ]:
import subprocess

subprocess.run(["make", "colab-check"], cwd=REPO_DIR, check=True)

### GPU identity check

Confirm this is the GPU you intended **before launching the campaign**. The cell below reads the
`run_metadata.json` written by the check run above (`environment` and `determinism` blocks).

In [ ]:
import glob
import json

hits = sorted(glob.glob(f"results/colab_check/{DATASET}/**/run_metadata.json", recursive=True))
assert hits, f"no run_metadata.json under results/colab_check/{DATASET}/ -- did `make colab-check` finish?"
meta = json.load(open(hits[-1]))
env = meta["environment"]
gpus = env.get("gpus") or []
assert gpus, "no GPU recorded in run_metadata.json -- is this a GPU runtime?"
assert meta["config"]["dataset"]["data_dir"] == f"DATA/{DATASET}/", meta["config"]["dataset"]["data_dir"]
gpu = gpus[0]
print("GPU name:      ", gpu.get("name"))
print("GPU memory GB: ", gpu.get("total_memory_gb"))
print("driver:        ", gpu.get("driver_version"))
print("CUDA (torch):  ", env["torch"].get("cuda"), "| cuDNN:", env["torch"].get("cudnn"))
print("determinism:   ", meta["determinism"])

In [ ]:
!poetry run python tools/loader.py --model results/colab_check/daninhas_full_fixed/SEED_1/NQ_100_NIL_100_NR_1_NE_10/SSRAEKmeansHCSampling/saved_model.pth

## 6. The clean campaign on `daninhas_full_fixed` (ADR 0016)

Everything the three papers need, re-run on the duplicate-free dataset: 459 runs / 153 run groups over
two manifests (main campaign 294 / 98 into `results/campaign_fixed/`, KMH sweep 165 / 55 into `results/campaign_kmh_fixed/`),
split into two GPU parts of about equal estimated GPU time, one per Colab account:

| GPU part | Account | Runs | Run groups |
|---|---|---|---|
| `part_1_gpu` | A | 282 | 94 |
| `part_2_gpu` | B | 177 | 59 |

Inside a part the order is: cheap paper-1 groups, the ablations, the KMH sweep, then the long baselines
(`AdversarialBIM`, `VAAL`/`WAAL`, `AdversarialDeepFool`) last. Re-running the run cell resumes (complete
leaves are skipped). Estimated hours per part: printed by the list cell. See `COLAB_RUNBOOK.md` section 6.

In [ ]:
!make campaign-list-gpu-part GPU_PART={GPU_PART} 2>&1 | tail -n 40

Run (skip-existing: **re-running this cell resumes after a disconnect** -- redo sections 0-5 first).

In [ ]:
import re
import subprocess

# Pre-run gate: every failure RAISES, so "Run all" stops here instead of launching a wrong part.
listing = subprocess.run(["make", "campaign-list-gpu-part", f"GPU_PART={GPU_PART}"], cwd=REPO_DIR,
                         check=True, capture_output=True, text=True).stdout
combined = listing.split("Combined:")[-1]
total = re.search(r"^\s*TOTAL\s+(\d+) runs \((\d+) run groups\)", combined, flags=re.M)
assert total, "campaign-list printed no TOTAL line:\n" + listing[-2000:]
assert (int(total.group(1)), int(total.group(2))) == (EXPECTED_RUNS, EXPECTED_GROUPS), (
    f"{GPU_PART}: campaign-list reports {total.group(1)} runs ({total.group(2)} run groups), "
    f"expected {EXPECTED_RUNS} ({EXPECTED_GROUPS})"
)
# Dirty state: this account's results must not hold leaves of the OTHER GPU part.
other = subprocess.run(["make", "campaign-verify-gpu-part", f"GPU_PART={OTHER_GPU_PART}"], cwd=REPO_DIR,
                       capture_output=True, text=True).stdout
found = re.search(r"Totals: OK=(\d+) INCOMPLETE=(\d+)", other)
assert found, "campaign-verify printed no Totals line:\n" + other[-2000:]
if int(found.group(1)) + int(found.group(2)):
    raise RuntimeError(f"results of {OTHER_GPU_PART} are on this Drive ({found.group(0)}): wrong account or "
                       "wrong GPU_PART? Stop and check before spending credits.")
print(total.group(0).strip(), f"-- gate OK for {GPU_PART}. The next cell launches the runs.")

In [ ]:
import time

start = time.time()
!make campaign-run-gpu-part GPU_PART={GPU_PART}

In [ ]:
elapsed_hours = (time.time() - start) / 3600
print(f"{GPU_PART}: this session ran {elapsed_hours:.2f} hours.")

**Progress** (usable anytime, e.g. after a disconnect):

In [ ]:
!make campaign-verify-gpu-part GPU_PART={GPU_PART} 2>&1 | grep -E "^Totals|seed [0-9]"
for root in RESULTS_ROOTS:
    log = f"{REPO_DIR}/{root}/campaign.log"
    tail = open(log).read().splitlines()[-5:] if os.path.exists(log) else ["no campaign.log yet"]
    print(f"--- {root}/campaign.log (last lines)", *tail, sep="\n")

In [ ]:
for root in RESULTS_ROOTS:
    log = f"{REPO_DIR}/{root}/failures.log"
    print(open(log).read() if os.path.exists(log) else f"{root}: no failures logged")

**Verify**: OK/INCOMPLETE/MISSING per job plus the seed-consistency audit (every run of a seed must start
from the identical initial labeled set). Exit code 0 = all OK, 1 = an incomplete/missing job or a FAIL
audit, 2 = only audit WARNs. Re-run the run cell to redo failed or missing jobs.

In [ ]:
!make campaign-verify-gpu-part GPU_PART={GPU_PART}

**Zip this GPU part** for download (both results roots, this account's `campaign.log` / `failures.log`
renamed to `campaign_<GPU_PART>.log` / `failures_<GPU_PART>.log` so the two accounts never collide). The
cell refuses (raises) unless verify reports every run of the part `OK` (`ZIP_PARTIAL = True` ships an
incomplete part on purpose) and checks the free space first: the checkpoints do not compress, so the
zip is about as large as the results (~95 MB per run).

In [ ]:
import re
import shutil
import subprocess

ZIP_PARTIAL = False  # True only to ship an incomplete part on purpose (e.g. no credits left)
verify = subprocess.run(["make", "campaign-verify-gpu-part", f"GPU_PART={GPU_PART}"], cwd=REPO_DIR,
                        capture_output=True, text=True)
totals = re.search(r"Totals: OK=(\d+) .*", verify.stdout)
print(totals.group(0) if totals else verify.stdout[-2000:] + verify.stderr[-2000:])
n_ok = int(totals.group(1)) if totals else 0
if n_ok != EXPECTED_RUNS and not ZIP_PARTIAL:
    raise RuntimeError(f"verify reports OK={n_ok}, expected {EXPECTED_RUNS}: re-run the run cell (resumes) first")
zip_path = f"/content/{GPU_PART}.zip"
need = sum(os.path.getsize(os.path.join(d, f)) for root in RESULTS_ROOTS
           for d, _, files in os.walk(f"{REPO_DIR}/{root}") for f in files)
free = min(shutil.disk_usage("/content").free, shutil.disk_usage(DRIVE_ROOT).free)
print(f"results to zip: {need / 1e9:.1f} GB; free (local disk / Drive, the smaller): {free / 1e9:.1f} GB")
if free < 1.1 * need:
    raise RuntimeError("Not enough free space for the zip: free Drive space, or download the two results "
                       "folders from Drive directly (COLAB_RUNBOOK.md section 6, step 9).")
excluded = " ".join(f"-x '{root}/campaign.log' -x '{root}/failures.log'" for root in RESULTS_ROOTS)
subprocess.run(f"rm -f {zip_path} && zip -qr {zip_path} {' '.join(RESULTS_ROOTS)} {excluded}",
               shell=True, cwd=REPO_DIR, check=True)
stage = f"/content/zip_stage_{GPU_PART}"
shutil.rmtree(stage, ignore_errors=True)
for root in RESULTS_ROOTS:
    os.makedirs(f"{stage}/{root}")
    for name in ("campaign", "failures"):
        log = f"{REPO_DIR}/{root}/{name}.log"
        if os.path.exists(log):
            shutil.copy(log, f"{stage}/{root}/{name}_{GPU_PART}.log")
subprocess.run(f"cd {stage} && zip -qr {zip_path} results", shell=True, check=True)
listing = subprocess.run(["unzip", "-l", zip_path], capture_output=True, text=True, check=True).stdout
n_models = listing.count("saved_model.pth")
print(f"{zip_path}: {n_models} checkpoints")
if n_models != (n_ok if ZIP_PARTIAL else EXPECTED_RUNS):
    raise RuntimeError(f"zip holds {n_models} checkpoints, expected {n_ok if ZIP_PARTIAL else EXPECTED_RUNS}")
subprocess.run(["cp", zip_path, f"{DRIVE_ROOT}/"], check=True)
subprocess.run(["ls", "-lh", f"{DRIVE_ROOT}/{GPU_PART}.zip"], check=True)

**Optional: partial report** of this account's part (the tables of the full campaign are built locally
after merging both parts, `COLAB_RUNBOOK.md` section 6, step 10). Rows of the other part show `TBD`.

In [ ]:
!make campaign-report-all

## 7. Troubleshooting

See `COLAB_RUNBOOK.md` section 7. Quick pointers:

- Session dropped / limit reached -> new runtime, Parameters cell unchanged, sections 0-5, then the run
  cell again (complete leaves are skipped; at most the run in progress is lost).
- Drive FUSE `Transport endpoint is not connected` -> re-run the mount cell and `make colab-setup`.
- `results/` symlink conflicts -> read `scripts/colab/setup_colab.sh`'s error message (it names the path).
- Out of GPU memory -> stop and report it; never edit a committed params file to lower the batch size.